# فاز ۳: آموزش، استقرار و ارزیابی مدل سبک و مقرون‌به‌صرفه (Edge AI)
### Phase 3: Cost-Effective Model Training, Efficiency Profiling & Benchmarking

در این بخش، معماری بهینه‌شده **Cost-Effective U-Net** با کانولوشن‌های تفکیک‌پذیر عمقی (Depthwise Separable) آموزش داده شده و کارایی محاسباتی آن (حجم، پارامتر، زمان استنتاج و نرخ فریم) با مدل‌های سنگین ادبیات علمی مقایسه می‌گردد.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.config import PROJECT_ROOT, NUM_CLASSES, CHECKPOINTS_DIR
from src.models.lightweight_unet import CostEffectiveMultiTaskWoundUNet
from src.utils.metrics import profile_model_efficiency
from src.evaluate import evaluate_and_benchmark

print("Modules loaded successfully!")

## ۱. بررسی معماری مدل و تعداد پارامترها (سبک‌سازی برای لبه)
مدل ما به جای کانولوشن‌های استاندارد سنگین، از کانولوشن‌های تفکیک‌پذیر عمقی استفاده می‌کند:

In [ ]:
model = CostEffectiveMultiTaskWoundUNet(in_channels=3, num_classes=NUM_CLASSES, base_channels=32)
eff = profile_model_efficiency(model, input_size=(1, 3, 256, 256))

print(f"تعداد کل پارامترها: {eff['total_params']:,} (~{eff['total_params']/1e6:.2f} میلیون)")
print(f"حجم فایل مدل در حافظه: {eff['param_size_mb']:.2f} مگابایت")
print(f"زمان استنتاج میانگین روی CPU: {eff['mean_latency_ms']:.2f} میلی‌ثانیه")
print(f"نرخ پردازش لحظه‌ای: {eff['fps']:.1f} فریم بر ثانیه")

## ۲. نمودار منحنی‌های یادگیری (Loss & Validation Dice Score)

In [ ]:
curve_path = PROJECT_ROOT / "results/training_curves.png"
if curve_path.exists():
    curve_img = plt.imread(str(curve_path))
    plt.figure(figsize=(12, 5), dpi=150)
    plt.imshow(curve_img)
    plt.axis('off')
    plt.title("Training Loss & Validation Metrics Progression", fontsize=12, fontweight='bold')
    plt.show()

## ۳. مقایسه کیفی خروجی مدل (Ground Truth vs Model Prediction)

In [ ]:
pred_path = PROJECT_ROOT / "results/qualitative_predictions.png"
if pred_path.exists():
    pred_img = plt.imread(str(pred_path))
    plt.figure(figsize=(12, 8), dpi=150)
    plt.imshow(pred_img)
    plt.axis('off')
    plt.title("Qualitative Segmentation Comparison on Validation Patients", fontsize=12, fontweight='bold')
    plt.show()